# GPU Feature Discovery

A practical reference for **GPU Feature Discovery (GFD)** — the NVIDIA component
that automatically labels Kubernetes nodes with the GPU hardware and software
features they expose (product name, count, memory, compute capability, driver
and CUDA versions, MIG profiles, time-slicing replicas). Those labels let the
scheduler place workloads on exactly the right GPUs via `nodeSelector` and node
affinity.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

**GPU Feature Discovery** (`gpu-feature-discovery`, GFD) is a NVIDIA Kubernetes
component that detects the GPUs on each node via **NVML** and publishes their
attributes as **node labels** under the `nvidia.com/` prefix. It is built to
work alongside **Node Feature Discovery (NFD)** — GFD produces the GPU-specific
labels, NFD's worker applies them (and the generic CPU/kernel/PCI labels) to the
node object.

GFD runs as a **DaemonSet** on GPU nodes. On an interval (default 60s) it reads
GPU properties and either writes them to a feature file that the co-located
NFD worker reads (`/etc/kubernetes/node-feature-discovery/features.d/`), or — in
recent versions with `--use-node-feature-api` — applies them directly through
NFD's **NodeFeature CRD**. The result is labels like:

```
nvidia.com/gpu.present=true
nvidia.com/gpu.count=8
nvidia.com/gpu.product=NVIDIA-A100-SXM4-80GB
nvidia.com/gpu.memory=81920
nvidia.com/gpu.compute.major=8
nvidia.com/cuda.driver-version.full=535.104.05
nvidia.com/mig.capable=true
```

GFD is one of the components installed and managed by the **NVIDIA GPU
Operator**; you rarely deploy it by hand on a managed cluster.

### Why use it?

- **GPU-aware scheduling** — target a specific GPU model, memory size, compute
  capability, or MIG profile with a plain `nodeSelector`/affinity, instead of
  hard-coding node names.
- **Heterogeneous fleets** — in a cluster mixing T4, L4, A100, and H100 nodes,
  route inference to cheap GPUs and fp64/training to the big ones automatically.
- **MIG and time-slicing visibility** — exposes MIG profile counts and shared
  GPU replicas as labels so schedulers and humans can reason about sliced GPUs.
- **Zero app instrumentation** — labels are derived from the hardware; you don't
  change your training/inference code.

### When to use it?

- You run GPUs on Kubernetes and need workloads pinned to particular GPU types.
- You operate a mixed-GPU cluster and want the scheduler, not operators, to make
  placement decisions.
- You use MIG or time-slicing and want the partitioning reflected in labels.

## Key Features

### Core capabilities of GPU Feature Discovery

| Capability | Description | Why it matters |
|---|---|---|
| NVML-based discovery | Reads GPU product, count, memory, compute capability, driver/CUDA versions | Accurate, vendor-sourced labels with no manual upkeep |
| NFD integration | Emits GPU labels through Node Feature Discovery (feature file or NodeFeature CRD) | One labeling pipeline for both generic and GPU features |
| MIG strategy support | `none` / `single` / `mixed` change how MIG devices are labeled | Schedule onto specific MIG profiles (`nvidia.com/mig-1g.10gb.count`) |
| Time-slicing labels | Adds `gpu.replicas`, `gpu.sharing-strategy`, and a `-SHARED` product suffix | Makes shared/over-subscribed GPUs visible to scheduling |
| Periodic relabeling | Re-runs on `--sleep-interval` (default 60s); `--oneshot` to label once | Picks up driver upgrades / MIG reconfiguration without restarts |
| GPU Operator managed | Shipped and version-matched by the NVIDIA GPU Operator | Consistent with driver, device plugin, DCGM exporter |
| Timestamp label | `nvidia.com/gfd.timestamp` marks the last labeling pass | Confirms GFD is alive and labels are fresh |

## Architecture Overview

```text
        +------------------------ GPU node ------------------------+
        |                                                          |
        |   NVIDIA driver + NVML                                   |
        |          ^                                               |
        |          | read GPU product/count/memory/CC/MIG          |
        |   +------+----------------------------+                  |
        |   |   gpu-feature-discovery (DaemonSet)|                 |
        |   |   - applies --mig-strategy          |                |
        |   |   - writes feature file  OR         |                |
        |   |     NodeFeature CRD (--use-node-...) |               |
        |   +------+------------------------------+                |
        |          |                                               |
        |   +------v---------------+   (legacy: shared hostPath)   |
        |   |  nfd-worker          |---features.d/ file----+       |
        |   +------+---------------+                        |      |
        +----------|---------------------------------------|-------+
                   | gRPC / NodeFeature                     |
                   v                                        v
            nfd-master  ----- patches ----->  Node object labels
                                              (nvidia.com/gpu.*)
                                                     |
                                                     v
                          kube-scheduler uses nodeSelector / affinity
```

### Components

1. **NVIDIA driver + NVML** — the source of truth GFD queries for GPU
   properties.
2. **gpu-feature-discovery (DaemonSet)** — runs on each GPU node, generates the
   `nvidia.com/*` GPU labels according to the configured MIG strategy.
3. **Node Feature Discovery (NFD)** — `nfd-worker` collects features (including
   GFD's) on each node; `nfd-master` writes them as labels on the Node object.
   GFD requires NFD to be present.
4. **NodeFeature CRD path** — with `--use-node-feature-api=true`, GFD creates a
   `NodeFeature` object that NFD reconciles, replacing the older shared
   feature-file approach.
5. **kube-scheduler** — consumes the resulting labels through `nodeSelector` /
   `nodeAffinity` to place pods on matching GPUs.

## Installation

### Prerequisites

- A Kubernetes cluster with **NVIDIA GPU nodes** and the **NVIDIA driver**
  installed (or managed by the GPU Operator).
- **NVIDIA Container Toolkit** so the GFD pod can access the GPU via NVML.
- **Node Feature Discovery (NFD)** deployed in the cluster — GFD depends on it
  to apply labels. The GPU Operator and the GFD Helm chart can install NFD for
  you.
- For MIG labels: a **MIG-capable GPU** (A100/H100) with MIG enabled.

> GFD is a **container/DaemonSet, not a pip package** — there is nothing to
> `pip install`. Since k8s-device-plugin **v0.15.0**, GFD was merged into the
> `NVIDIA/k8s-device-plugin` repository and ships from the same image; the
> standalone `NVIDIA/gpu-feature-discovery` repo is deprecated. The cell below
> shows the real ways to deploy it.

In [ ]:
%%bash
# GPU Feature Discovery is deployed as a Kubernetes DaemonSet, not via pip.

# 1) RECOMMENDED: let the NVIDIA GPU Operator manage GFD (plus driver,
#    container toolkit, device plugin, DCGM exporter, and NFD) together.
helm repo add nvidia https://helm.ngc.nvidia.com/nvidia
helm repo update
helm install --wait gpu-operator nvidia/gpu-operator \
  --namespace gpu-operator --create-namespace

# 2) Standalone Helm chart (installs NFD as a dependency unless disabled).
helm repo add nvgfd https://nvidia.github.io/k8s-device-plugin
helm repo update
helm install gpu-feature-discovery nvgfd/gpu-feature-discovery \
  --namespace gpu-feature-discovery --create-namespace \
  --set migStrategy=mixed

# 3) Verify the GFD pods are running, one per GPU node.
kubectl get pods -n gpu-operator -l app=gpu-feature-discovery -o wide


## Basic Usage

### Quick start

Once GFD (and NFD) are running, the GPU labels appear on each GPU node. Inspect
them with `kubectl`, then use them in a `nodeSelector` to pin a pod to a specific
GPU model. The labels are the whole point — everything else builds on them.

In [ ]:
%%bash
# Show every nvidia.com/* label GFD/NFD applied to the GPU nodes.
kubectl get nodes -o json | jq '.items[] | {
  node: .metadata.name,
  gpu_labels: (.metadata.labels | with_entries(select(.key | startswith("nvidia.com/"))))
}'

# Or filter from the command line for one node:
kubectl get node <gpu-node> -o jsonpath='{.metadata.labels}' | tr ',' '\n' | grep nvidia.com

# Typical labels on an 8x A100 (80GB) node:
#   nvidia.com/gpu.present=true
#   nvidia.com/gpu.count=8
#   nvidia.com/gpu.product=NVIDIA-A100-SXM4-80GB
#   nvidia.com/gpu.memory=81920
#   nvidia.com/gpu.family=ampere
#   nvidia.com/gpu.compute.major=8
#   nvidia.com/gpu.compute.minor=0
#   nvidia.com/cuda.driver-version.full=535.104.05
#   nvidia.com/cuda.runtime-version.full=12.2
#   nvidia.com/mig.capable=true
#   nvidia.com/gfd.timestamp=1718900000


In [ ]:
# Reference: the GPU labels GFD emits and what they mean.
# (Pure reference table -- no cluster or GPU required to read it.)
labels = {
    "nvidia.com/gpu.present":            "true on any node with a GPU",
    "nvidia.com/gpu.count":              "number of GPUs (or MIG devices) on the node",
    "nvidia.com/gpu.product":            "sanitized model, e.g. NVIDIA-A100-SXM4-80GB",
    "nvidia.com/gpu.memory":             "per-GPU framebuffer memory in MiB",
    "nvidia.com/gpu.family":             "architecture family, e.g. ampere / hopper",
    "nvidia.com/gpu.compute.major":      "CUDA compute capability major (8 for A100)",
    "nvidia.com/gpu.compute.minor":      "CUDA compute capability minor (0 for A100)",
    "nvidia.com/gpu.machine":            "system/board model name",
    "nvidia.com/cuda.driver-version.full":  "installed driver version, e.g. 535.104.05",
    "nvidia.com/cuda.runtime-version.full": "CUDA runtime version, e.g. 12.2",
    "nvidia.com/mig.capable":            "true if the GPU supports MIG",
    "nvidia.com/mig.strategy":           "none / single / mixed",
    "nvidia.com/gpu.replicas":           "time-slicing replicas advertised per GPU",
    "nvidia.com/gpu.sharing-strategy":   "time-slicing / mps when sharing is on",
    "nvidia.com/gfd.timestamp":          "unix time of the last labeling pass",
}
width = max(len(k) for k in labels)
for name, desc in labels.items():
    print(f"{name:<{width}}  {desc}")


## Advanced Features

### 1. MIG strategies (`--mig-strategy`)

How MIG-partitioned GPUs are labeled depends on the strategy:

- **`none`** — MIG ignored; the GPU is labeled as a whole device.
- **`single`** — the node is assumed homogeneous; `nvidia.com/gpu.product` gets
  a MIG suffix (e.g. `NVIDIA-A100-SXM4-80GB-MIG-1g.10gb`) and `gpu.count`
  reflects the number of MIG instances.
- **`mixed`** — each MIG profile is exposed independently with its own labels:
  `nvidia.com/mig-1g.10gb.count`, `nvidia.com/mig-1g.10gb.memory`,
  `nvidia.com/mig-1g.10gb.engines.copy`, etc., so different profiles on one node
  can be scheduled separately.

### 2. Time-slicing labels

When the device plugin is configured for **time-slicing**, GFD reflects it:
`nvidia.com/gpu.replicas` shows how many shared replicas each GPU advertises,
`nvidia.com/gpu.sharing-strategy=time-slicing`, and the product label gains a
`-SHARED` suffix (e.g. `NVIDIA-A100-SXM4-80GB-SHARED`) so you can distinguish
dedicated from oversubscribed GPUs in selectors.

### 3. NodeFeature API mode (`--use-node-feature-api`)

Newer GFD/NFD can skip the shared hostPath feature file and instead have GFD
create a **`NodeFeature`** custom resource that `nfd-master` reconciles into
labels. This decouples GFD from the NFD worker's filesystem and is the direction
the project is moving.

### 4. Operational flags

`--oneshot` labels once and exits (useful in init containers/jobs);
`--sleep-interval` controls the relabel period; `--no-timestamp` drops the
`gfd.timestamp` label; `--output-file` sets the feature-file path in legacy
mode.

In [ ]:
# A GPU Operator ClusterPolicy fragment toggling GFD options, and the
# device-plugin time-slicing config that GFD reflects into labels.
gfd_clusterpolicy = '''apiVersion: nvidia.com/v1
kind: ClusterPolicy
metadata:
  name: cluster-policy
spec:
  gfd:
    enabled: true
    # image/version are pinned by the operator release
  mig:
    strategy: mixed            # none | single | mixed -> drives MIG labels
'''

time_slicing_cm = '''apiVersion: v1
kind: ConfigMap
metadata:
  name: time-slicing-config
  namespace: gpu-operator
data:
  any: |-
    version: v1
    sharing:
      timeSlicing:
        resources:
          - name: nvidia.com/gpu
            replicas: 4        # GFD -> nvidia.com/gpu.replicas=4, product gains -SHARED
'''

print(gfd_clusterpolicy)
print(time_slicing_cm)


## Use Cases

#### Pin a workload to a specific GPU model
- **Context:** A cluster mixes T4, L4, and A100 nodes; a training job needs an
  A100.
- **Implementation:** Add `nodeSelector: { nvidia.com/gpu.product:
  NVIDIA-A100-SXM4-80GB }` plus the `nvidia.com/gpu` resource request.
- **Results:** The scheduler only considers A100 nodes; cheaper GPUs stay free
  for inference.

#### Route by compute capability or memory
- **Context:** An fp64 / large-model job needs compute capability ≥ 8.0 and
  40GB+ of VRAM.
- **Implementation:** Node affinity on `nvidia.com/gpu.compute.major` and
  `nvidia.com/gpu.memory` (with `Gt` operators via numeric labels).
- **Results:** Hardware-aware placement without hard-coding node names.

#### Schedule onto a specific MIG profile
- **Context:** A100s are MIG-partitioned into `1g.10gb` and `3g.40gb` slices.
- **Implementation:** With `mig-strategy=mixed`, request the MIG resource
  (`nvidia.com/mig-1g.10gb`) and select on `nvidia.com/mig-1g.10gb.count`.
- **Results:** Small inference pods land on small slices; big jobs on large ones.

#### Distinguish dedicated vs time-sliced GPUs
- **Context:** Some nodes share GPUs via time-slicing for dev notebooks.
- **Implementation:** Select `nvidia.com/gpu.product` without the `-SHARED`
  suffix for latency-sensitive serving; allow `-SHARED` for batch/dev.
- **Results:** Production inference avoids oversubscribed GPUs.

In [ ]:
# Example pod specs that consume GFD labels for placement.
nodeselector_pod = '''apiVersion: v1
kind: Pod
metadata:
  name: a100-training
spec:
  nodeSelector:
    nvidia.com/gpu.product: NVIDIA-A100-SXM4-80GB   # exact, sanitized model string
  containers:
    - name: trainer
      image: nvcr.io/nvidia/pytorch:24.05-py3
      resources:
        limits:
          nvidia.com/gpu: 1
'''

affinity_pod = '''apiVersion: v1
kind: Pod
metadata:
  name: fp64-job
spec:
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
          - matchExpressions:
              - key: nvidia.com/gpu.compute.major
                operator: In
                values: ["8", "9"]          # Ampere/Hopper or newer
              - key: nvidia.com/gpu.memory
                operator: Gt
                values: ["40000"]           # > ~40 GiB framebuffer
  containers:
    - name: job
      image: my-fp64-workload:latest
      resources:
        limits:
          nvidia.com/gpu: 1
'''

print(nodeselector_pod)
print(affinity_pod)


## Best Practices

1. **Let the GPU Operator manage GFD.** On a managed cluster, run GFD through
   the operator so its version stays matched to the driver, device plugin, and
   NFD; avoid a second standalone GFD DaemonSet.
2. **Install NFD first (or let the chart do it).** GFD has no effect without NFD
   to apply the labels — confirm NFD is healthy before debugging "missing
   labels".
3. **Prefer node affinity over `nodeSelector` for soft constraints.** Use
   `preferredDuringScheduling` so pods still schedule when the ideal GPU is busy.
4. **Treat `gpu.product` as an exact, sanitized string.** Spaces become dashes
   and time-slicing/MIG add suffixes — copy the label value verbatim from
   `kubectl`, don't guess it.
5. **Keep MIG strategy consistent** across the device plugin and GFD (both
   `mixed`, or both `single`) so resource names and labels agree.
6. **Pin chart/operator versions.** Reproducible deployments avoid surprise
   label or behavior changes across releases.
7. **Alert on stale `gfd.timestamp`.** A timestamp that stops advancing means
   GFD isn't relabeling — catch it before scheduling decisions go stale.

## Common Pitfalls

1. **No `nvidia.com/*` labels appear.** The usual cause is **NFD not installed**
   (or unhealthy). GFD generates features but NFD applies them — without NFD,
   nothing lands on the node.
2. **`nodeSelector` never matches.** The `gpu.product` value is sanitized
   (spaces → dashes) and may carry `-MIG-...` or `-SHARED` suffixes; an
   approximate string silently matches nothing and the pod stays `Pending`.
3. **MIG labels missing.** MIG isn't enabled on the GPU, or `--mig-strategy` is
   `none` — switch to `single`/`mixed` and enable MIG via the operator.
4. **Time-slicing breaks existing selectors.** Turning on time-slicing appends
   `-SHARED` to `gpu.product`, so selectors written for the dedicated name stop
   matching.
5. **Running standalone GFD alongside the GPU Operator.** Two GFD DaemonSets
   fight over labels; pick one path.
6. **Numeric comparisons on string labels.** `Gt`/`Lt` affinity operators need
   labels that parse as integers — `gpu.memory` and `compute.major` are numeric,
   but `gpu.product` is not.

## Performance Optimization

### Tuning relabel cost

GFD is extremely lightweight — it samples NVML metadata, not live telemetry —
so its overhead is dominated by the relabel interval and the size of the label
set rather than by GPU work.

- **`--sleep-interval`:** the main knob. The default 60s is fine for almost
  everyone; only shorten it if you reconfigure MIG/drivers frequently and need
  labels to converge faster. Longer intervals reduce API churn on very large
  clusters.
- **`--oneshot`:** for immutable nodes whose GPU layout never changes, label
  once at boot and skip the periodic loop entirely.
- **MIG `mixed` cardinality:** `mixed` strategy multiplies labels per profile;
  on dense, many-profile nodes that is more labels for NFD to reconcile. Use
  `single` when a node is homogeneous.
- **NFD master load:** GFD's cost shows up as NodeFeature/label updates that
  `nfd-master` reconciles — align GFD's interval with NFD's resync to avoid
  redundant churn.

In [ ]:
# Rough estimate of GFD's labeling footprint as a function of cluster size
# and MIG strategy. (Illustrative arithmetic, not a benchmark.)
def gfd_footprint(num_gpu_nodes, mig_profiles=0, sleep_interval_s=60):
    # ~12 base GPU labels per node, plus ~5 labels per distinct MIG profile (mixed)
    labels_per_node = 12 + 5 * mig_profiles
    total_labels = labels_per_node * num_gpu_nodes
    relabels_per_hour = (3600 / sleep_interval_s) * num_gpu_nodes
    return labels_per_node, total_labels, int(relabels_per_hour)

for desc, (nodes, profiles, interval) in {
    "50 nodes, no MIG, 60s":        (50, 0, 60),
    "200 nodes, no MIG, 60s":       (200, 0, 60),
    "200 nodes, MIG mixed x3, 300s": (200, 3, 300),
}.items():
    per, total, rate = gfd_footprint(nodes, profiles, interval)
    print(f"{desc:30s} labels/node={per:3d}  total_labels={total:5d}  relabels/hr={rate}")


## Production Deployment

### DaemonSet on GPU nodes (managed by the GPU Operator)

In production GFD is normally a DaemonSet the **GPU Operator** deploys, scheduled
only onto GPU nodes (it tolerates the `nvidia.com/gpu` taint and selects the
operator's `gpu-feature-discovery` node label). The manifest below shows the
essential pieces if you run it standalone: GPU-node selection, NVML access, the
MIG strategy, and the NodeFeature API flag.

```yaml
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: gpu-feature-discovery
  namespace: gpu-feature-discovery
  labels: { app: gpu-feature-discovery }
spec:
  selector:
    matchLabels: { app: gpu-feature-discovery }
  template:
    metadata:
      labels: { app: gpu-feature-discovery }
    spec:
      nodeSelector:
        nvidia.com/gpu.present: "true"      # only GPU nodes (set by NFD/operator)
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule
      containers:
        - name: gpu-feature-discovery
          image: nvcr.io/nvidia/k8s-device-plugin:v0.15.0   # GFD ships in this image
          command: ["gpu-feature-discovery"]
          args:
            - --mig-strategy=mixed
            - --use-node-feature-api=true
            - --sleep-interval=60s
          securityContext:
            privileged: true                # NVML access
      # In legacy (feature-file) mode, mount the shared NFD features.d dir:
      #   volumeMounts: [{ name: features, mountPath: /etc/kubernetes/node-feature-discovery/features.d }]
      #   volumes: [{ name: features, hostPath: { path: /etc/kubernetes/node-feature-discovery/features.d } }]
```

NFD must be running in the cluster for these labels to reach the Node object.

## Monitoring and Observability

### What to watch

- **Label freshness:** `nvidia.com/gfd.timestamp` should advance every
  `sleep-interval`. A frozen timestamp means GFD stopped relabeling.
- **DaemonSet health:** one Ready GFD pod per GPU node
  (`kubectl get ds -n gpu-operator`); `CrashLoopBackOff` usually points at NVML
  / driver access problems.
- **Label correctness:** the GPU count and product on the node match the
  physical hardware and the device plugin's advertised `nvidia.com/gpu`
  capacity.
- **NFD pipeline:** `nfd-worker`/`nfd-master` logs — if NFD is down, GFD's
  features never become labels.

### Useful checks

```bash
# GFD DaemonSet rollout status and per-node pods.
kubectl get ds -n gpu-operator gpu-feature-discovery
kubectl get pods -n gpu-operator -l app=gpu-feature-discovery -o wide

# Confirm the timestamp label is recent across all GPU nodes.
kubectl get nodes -L nvidia.com/gpu.product,nvidia.com/gfd.timestamp

# Tail GFD logs on a node having labeling issues.
kubectl logs -n gpu-operator -l app=gpu-feature-discovery --tail=50
```

GFD does not export Prometheus metrics itself — pair it with **DCGM Exporter**
for GPU telemetry; GFD's job is labeling, not runtime monitoring.

## Troubleshooting

#### No `nvidia.com/*` labels on GPU nodes
- **Symptoms:** `kubectl get node -o jsonpath='{.metadata.labels}'` shows no
  `nvidia.com/gpu.*`.
- **Cause:** NFD is not installed/healthy, or GFD can't reach the GPU.
- **Solution:** Verify NFD pods are Running, confirm the GFD DaemonSet is Ready
  on the node, and check `nvidia-smi` works on the host.

#### GFD pod is in `CrashLoopBackOff`
- **Symptoms:** Logs show NVML init / "no devices found" errors.
- **Cause:** Missing/incompatible driver, no NVIDIA Container Toolkit, or the
  pod lacks GPU access.
- **Solution:** Ensure the driver and container toolkit are installed (the GPU
  Operator does this), and that the pod is privileged / sees the GPU.

#### `nodeSelector` on `gpu.product` leaves pods `Pending`
- **Symptoms:** Pod never schedules though matching GPUs exist.
- **Cause:** The product string is wrong — wrong casing, spaces instead of
  dashes, or a missing `-MIG-...`/`-SHARED` suffix.
- **Solution:** Copy the exact label value from `kubectl get node <n>
  -o jsonpath='{.metadata.labels.nvidia\.com/gpu\.product}'`.

#### MIG labels don't appear
- **Symptoms:** No `nvidia.com/mig-*` labels on an A100/H100 node.
- **Cause:** MIG isn't enabled, or `--mig-strategy=none`.
- **Solution:** Enable MIG (operator `migManager`) and set the strategy to
  `single` or `mixed`; restart GFD.

## Comparison with Alternatives

| Aspect | GPU Feature Discovery | Manual node labels | Node Feature Discovery alone | NVIDIA Device Plugin |
|---|---|---|---|---|
| Source of GPU attributes | NVML (automatic) | Human-maintained | Generic CPU/kernel/PCI only | Advertises `nvidia.com/gpu` resource |
| GPU product / memory / CC labels | Yes | Error-prone, manual | No GPU-specific labels | No (only the resource count) |
| MIG / time-slicing labels | Yes | No | No | Exposes MIG/shared *resources*, not labels |
| Keeps labels current | Yes (periodic) | No, drifts | n/a for GPUs | n/a |
| Scheduling role | Labels for affinity/selectors | Same, if maintained | Generic features | Resource requests/limits |
| Setup effort | Low (GPU Operator) | High, fragile | Low (but no GPU detail) | Low (required regardless) |

### When to choose GPU Feature Discovery

- You need **fine-grained GPU placement** (by model, memory, compute capability,
  or MIG profile), not just "give me a GPU".
- You run a **heterogeneous** GPU fleet and want labels maintained automatically.
- You already use (or will deploy) **NFD** and the **GPU Operator**.

GFD complements — it does not replace — the **device plugin** (which advertises
the schedulable `nvidia.com/gpu` resource). You typically run both: the device
plugin for the resource, GFD for the descriptive labels.

## Resources

### Official documentation
- GFD / k8s-device-plugin repository (GFD merged in since v0.15.0):
  https://github.com/NVIDIA/k8s-device-plugin
- Deprecated standalone GFD repository:
  https://github.com/NVIDIA/gpu-feature-discovery
- NVIDIA GPU Operator docs:
  https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/index.html
- Node Feature Discovery docs: https://nfd.sigs.k8s.io/

### Tutorials and guides
- GPU Operator component overview (GFD section):
  https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-operator-mig.html
- MIG support in Kubernetes:
  https://docs.nvidia.com/datacenter/cloud-native/kubernetes/latest/index.html
- Time-slicing GPUs in Kubernetes:
  https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-sharing.html

### Community resources
- NVIDIA Cloud Native / k8s-device-plugin issues:
  https://github.com/NVIDIA/k8s-device-plugin/issues
- NVIDIA Developer Forums (datacenter/cloud-native):
  https://forums.developer.nvidia.com/c/datacenter/

### Related technologies
- NVIDIA Device Plugin for Kubernetes (advertises `nvidia.com/gpu`)
- Node Feature Discovery (applies the labels GFD generates)
- NVIDIA GPU Operator (deploys driver, toolkit, device plugin, GFD, DCGM)
- NVIDIA MIG Manager and time-slicing (reflected in GFD labels)